# In-context ASR: Self-Training Evaluation Demo

This notebook runs the `in-context-asr` evaluation on the LCASR model, both **without** and **with** per-utterance self-training (dynamic evaluation) adaptation, so you can compare the two on the same test set.

Self-training recipe is adapted from [robflynnyh/Self-Train-Before-You-Transcribe](https://github.com/robflynnyh/Self-Train-Before-You-Transcribe) (`dynamic_eval_ctc_loss`). The function now lives directly in this repo at `models/lcasr/dynamic_eval.py`.

Runs on Colab with a GPU runtime.

## 1. Clone repos and install dependencies

We need this repo (`in-context-asr`) for the evaluation harness and the data, and `long-context-asr` for the pretrained LCASR model and its building blocks (augmentation, optimiser, decoder).

In [ ]:
!git clone https://github.com/robflynnyh/in-context-asr
%cd in-context-asr
!git pull
%cd ..

In [ ]:
!git clone https://github.com/robflynnyh/long-context-asr
%cd long-context-asr
!git checkout v1.0
!pip install .
%cd ..

In [ ]:
!pip install einops omegaconf torch_ema
!pip install openai-whisper  # needed for EnglishTextNormalizer used in run_evaluation.py

## 2. Download the pretrained LCASR checkpoint

In [ ]:
%cd in-context-asr
!wget -nc https://huggingface.co/rjflynn2/lcasr-6L-768D-6H-RB-1p5M/resolve/main/n_seq_sched_16384_rp_1/step_105360.pt

## 3. Sanity-check: load the model and run a single clip through both pipelines

We import `load_model.load` from the lcasr wrapper, first without `--self_train` then with it, to confirm everything is wired up before the full eval sweep.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())  # so `from models.lcasr.load_model import ...` resolves

from models.lcasr.load_model import load as load_lcasr

class Args:
    checkpoint = 'step_105360.pt'
    name = 'SCConformerXL'
    self_train = False
    optim_lr = 9e-5
    epochs = 1
    shuffle = True
    spec_augment_n_time_masks = 0
    spec_augment_n_freq_masks = 6
    spec_augment_freq_mask_param = 34
    spec_augment_zero_masking = False
    verbose = False

pipeline_baseline = load_lcasr(Args())

sample = './data/1/sentence_without_repeat.wav'
print('Baseline (no adaptation):', pipeline_baseline(sample))

In [ ]:
class STArgs(Args):
    self_train = True
    epochs = 5  # paper default

pipeline_st = load_lcasr(STArgs())
print('With self-training:', pipeline_st(sample))

## 4. Run the full evaluation — baseline

Evaluates `correct_without_repeat`, `correct_in_corrupt_repeat`, and `correct_in_clear_repeat` across all 20 clips.

In [ ]:
!python run_evaluation.py --model lcasr --checkpoint step_105360.pt

## 5. Run the full evaluation — with self-training adaptation

Each clip is adapted for `--epochs` passes (default 5 in the paper) before its logits are decoded. Model parameters are restored between clips so there is no leakage across utterances.

In [ ]:
!python run_evaluation.py --model lcasr --checkpoint step_105360.pt --self_train --epochs 5

## 6. Compare

The final lines of each run give `correct_without_repeat`, `correct_in_corrupt_repeat`, and `correct_in_clear_repeat` percentages. Self-training should primarily help on noisier / harder clips where the model benefits from adapting to the recording's acoustic conditions.